# Class 12: Grouping and Pivoting
## Operating on Tables (not Operating Tables :-)

*Elements of Data Science — Honors*

The same 1,025 patients you wrote a diagnostic rule for in Class 09. Today you get two
methods that summarize a whole table at once — and then find out what they hide.

Work with your team. Blank cells are yours; filled cells are ours to run and discuss.
Record answers on the **Class 12 activity sheet**.

## The data

#### Metadata
From Kaggle, see [https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset).
More on the original study [here.](https://archive.ics.uci.edu/dataset/45/heart+disease)

1. `age`
2. `sex` — 1 = male, 0 = female
3. `cp` — chest pain type (4 values)
4. `trestbps` — resting blood pressure
5. `chol` — serum cholesterol in mg/dl
6. `fbs` — fasting blood sugar > 120 mg/dl (1 = yes, 0 = no)
7. `restecg` — resting electrocardiographic results (0, 1, 2)
8. `thalach` — maximum heart rate achieved
9. `exang` — exercise induced angina (1 = yes, 0 = no)
10. `oldpeak` — ST depression induced by exercise relative to rest
11. `slope` — slope of the peak exercise ST segment
12. `ca` — number of major vessels (0–3) colored by fluoroscopy
13. `thal` — thalassemia: 0 = normal, 1 = fixed defect, 2 = reversible defect

*target* — 0 = no heart disease, 1 = heart disease

**Keep this list where you can see it. One of today's jobs is to find out where it is
wrong.**

<img src="https://www.wikidoc.org/images/5/53/SinusRhythmLabels.png" alt="Labeled diagram of a normal sinus rhythm on an EKG trace" width=500, height="auto" class="blog-image">

In [ ]:
from datascience import *
import numpy as np
# import for plotting
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Fix for datascience plots
import collections as collections
import collections.abc as abc
collections.Iterable = abc.Iterable

In [ ]:
import os
path = 'data/heart.csv' if os.path.exists('data/heart.csv') else 'heart.csv'
heart = Table.read_table(path)
heart

---
# Part 1. The Long Way

Answer this with what you already have — `where`, `column`, `np.median`, `np.mean`.

**Q1.1** Split the patients at the median age into `younger` (at or below) and `older`
(above). Compare the two groups on average cholesterol and on average resting blood
pressure.

**Stop.** Count the lines of code your team wrote and record the number on the
worksheet.

---
# Part 2. `group()` — The Short Way

    Table.group(column_or_label, collect=None)

The column you group on becomes the rows — one row for each distinct value. `collect` is a
function applied to every other column, once per group. The default is to count.

In [ ]:
# Default: count the rows in each group
heart.group("age").sort('age')

In [ ]:
# Ask for the median instead
# We get a column back for every feature, because each has its own values
heart.group("age", np.median)

### Does median cholesterol rise with age in these patients?

In [ ]:
heart.group("age", np.median).select('age', 'chol median').sort('age').scatter('age', 'chol median', fit_line=True)

*Optional, if the chaining above is hard to read:* the same thing one step at a time.

In [ ]:
step1 = heart.group("age", np.median)     # group
step2 = step1.select('age', 'chol median')  # keep two columns
step3 = step2.sort('age')                   # put ages in order
step3.scatter('age', 'chol median', fit_line=True)

### Grouping on more than one column

In [ ]:
# One row for each unique combination
heart.group(["sex", "target"])

**Q2.1** Redo Q1.1 with `group()`. How many lines now?

*Hint: `group` needs a column to group on, so you will have to make one — a column saying
whether each patient is older than the median. A boolean array works.*

### Your own collect function

Any function that takes an array and returns one value will do.

In [ ]:
def value_range(x):
    return max(x) - min(x)

# Test it before applying it
value_range(make_array(2, 4, 7, 9, 1))

In [ ]:
heart.group("target", value_range)

**A note on naming.** The obvious name for that function is `range`, which is already
a Python built-in. Defining your own replaces it for the rest of the session, and any later
`range(10)` breaks in a way that is very hard to diagnose. There is a question about this on
the worksheet.

---
# Part 3. `pivot()` — Two Grouping Variables at Once

    Table.pivot(columns, rows, values=None, collect=None, zero=None)

Unique values of one column become the rows, unique values of another become the columns,
and each cell aggregates the rows matching both. Compare it with `heart.group(["sex",
"target"])` above — same information, different arrangement, and the arrangement is the
point.

**Q3.1** Before running the next cell: `slope` takes values 0, 1, 2 and `target` takes 0, 1.
Sketch the shape of the result on your worksheet — how many rows, how many columns, what is
in a cell.

In [ ]:
#------------columns---rows----
heart.pivot('slope', 'target')

In [ ]:
#--------------------------column----row----values---apply-to-values---
chol_by_age = heart.pivot('target', 'age', 'chol', collect=np.median)
chol_by_age.scatter('age')

Those zeros are not measurements. They are ages where one group has no patients at
all, and `pivot` filled the empty cell with zero. **Q3.2 on the worksheet:** why is that a
dangerous default?

In [ ]:
chol_by_age = chol_by_age.where('0', are.above(0)).where('1', are.above(0))
chol_by_age.scatter('age', fit_line=True, overlay=False)

---
# Part 4. The Cholesterol Paradox

Run this.

In [ ]:
heart.hist("chol", group='target', bins=20)

**A surprising result.** The patients *with* heart disease do not have higher
cholesterol here.

**Q4.1** Confirm it numerically — mean and median cholesterol for each value of `target`.
One line with `group()`.

**Q4.2** Write your team's explanation on the worksheet *before* running anything
else. Then test it.

The usual first hypothesis is composition. In Class 09 you found that women in this table
are far more likely to have heart disease than men, and women tend to have higher
cholesterol. If the healthy group is mostly women, that alone could produce the pattern.

In [ ]:
heart.group('sex', np.mean).select('sex', 'age mean', 'chol mean', 'target mean')

**Q4.3** Now break the comparison out by sex — group on **both** `sex` and `target`
and look at mean cholesterol in each of the four cells. Does controlling for sex make the
paradox go away, or make it worse?

**Q4.4** Record what happened, then argue it out with the other teams.

Two explanations we cannot test with these columns:

- **Treatment.** A patient diagnosed with heart disease is put on a statin. Cholesterol
  measured after diagnosis is cholesterol after treatment.
- **Selection.** Everyone here was referred for cardiac catheterization. The "no disease"
  group is not healthy people — it is people sick enough to be sent for an angiogram whose
  arteries turned out clear.

---
# Part 5. Do You Trust This Table?

Every number today — and every probability you computed in Class 09 — assumes the table says
what the metadata says it says.

The metadata gives `thal` three values and `ca` four.

In [ ]:
heart.group('thal')

In [ ]:
heart.group('ca')

And how many patients are there? Grouping on *every* column at once puts identical
rows in the same group.

In [ ]:
distinct = heart.group(list(heart.labels))

print('rows in the file:', heart.num_rows)
print('distinct rows:   ', distinct.num_rows)

counts = distinct.column('count')
print('each distinct patient appears between', int(min(counts)), 'and', int(max(counts)), 'times')

**Q5.1** The original UCI Cleveland study had 303 patients. In one sentence, what was
done to this file before it was posted to Kaggle?

**Q5.2** Duplicating each patient several times barely moves a mean, but it multiplies the
apparent sample size. Later in the semester you will judge a result by how much variation
chance alone could produce — and sample size is what sets that. What would this file do to
such a judgment?

---
If your team gets here with time left, the last two questions on the worksheet are the best
ones on it.